<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Week 5 Data Loading & Preprocessing
This cell loads the dataset and prepares the variables (`df`, `X`, `y`, `groups`) required for the validation audit.

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    from getpass import getpass
    hf_token = getpass('HF_Token: ')

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

FACT = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
DIM = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

query = f"""
WITH prior AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_prior30,
           SUM(gsc_clicks) AS gsc_clicks_prior30,
           AVG(gsc_avg_position) AS gsc_avg_position_prior30
    FROM {FACT}
    WHERE month = '2026-02'
    GROUP BY content_hash_id, client_hash_id
),
last AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_last30,
           SUM(gsc_clicks) AS gsc_clicks_last30
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    p.content_hash_id, p.client_hash_id,
    p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
    l.gsc_impressions_last30, l.gsc_clicks_last30,
    d.word_count, d.content_type, d.main_intent
FROM prior p
JOIN last l USING (content_hash_id, client_hash_id)
JOIN {DIM} d USING (content_hash_id)
WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
"""

df = con.sql(query).df()
print(f"Rows: {len(df):,}")

df['impr_change_pct'] = (df['gsc_impressions_last30'] - df['gsc_impressions_prior30']) / df['gsc_impressions_prior30'] * 100
df['click_change_pct'] = (df['gsc_clicks_last30'] - df['gsc_clicks_prior30']) / df['gsc_clicks_prior30'] * 100

def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df['pattern_group'] = df.apply(assign_pattern, axis=1)
df['gsc_ctr_prior30'] = df['gsc_clicks_prior30'] / df['gsc_impressions_prior30']

print(df['pattern_group'].value_counts())
print(f"\nBase rate (answered_away): {(df['pattern_group'] == 'answered_away').mean():.3f}")

y = (df['pattern_group'] == 'answered_away').astype(int)

feature_cols_numeric = [
    'gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
    'word_count', 'gsc_ctr_prior30'
]
feature_cols_categorical = ['content_type', 'main_intent']

X = pd.get_dummies(df[feature_cols_numeric + feature_cols_categorical], dummy_na=True)
X[feature_cols_numeric] = X[feature_cols_numeric].fillna(0)

groups = df['client_hash_id']

print(f"\nX shape: {X.shape}, y length: {len(y)}, groups length: {len(groups)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 29,516
pattern_group
stable_other     19186
answered_away     6759
normal_decay      3571
Name: count, dtype: int64

Base rate (answered_away): 0.229

X shape: (29516, 14), y length: 29516, groups length: 29516


## 1. Two paper findings + my methodology questions

**Finding #1 — "The Freshness Multiplier" (361+ day bucket, 283:1 growth ratio)**

Where does the number come from: the paper computes a growth-to-decline ratio per freshness
bucket, and the 361+ bucket shows 283 growing pages against just 1 declining page. My question:
with n=1 on the declining side, is 283:1 a stable measurement or is it one page away from being
a completely different number? The paper actually answers this itself in the same section,
noting the bucket is "tiny and unstable," which is exactly the kind of self-correction this
assignment asks me to practice — so my methodology question here is really "would I have caught
this without the paper flagging it for me," and honestly I'm not sure I would have on a first
pass. The stricter number to report is probably a confidence interval on that ratio, or just
dropping the headline figure and keeping the 31-90 day bucket's 7.88:1 as the real claim, which
is what the paper does.

**Finding #2 — "AI Model Performance" (OpenAI vs Gemini age-controlled cohorts)**

Where does the label come from: health score, which the paper is upfront is a FlyRank composite
(impressions + position + CTR + scroll depth), not an external outcome. My methodology question:
does the validation design support a claim about which AI provider produces better content, or
does it only support a claim about which provider's pages score higher on a metric that's
partly made of the same inputs (impressions, position) that also drive provider assignment
indirectly through publish timing and topic mix? The paper's own random forest appendix shows
Average Position and Impressions are 43% and 32% of what predicts health score, so two providers
differing on health score could partly just mean they differ on position and impressions, which
is closer to circular than causal. The paper does hedge this correctly ("not a victory lap for
one provider family"), so my question is more about whether a reader skimming just the chart
would pick up that hedge, or just see "Gemini leads" and stop there.

## 2. My model under an honest split (before/after)

Before/after: random split vs grouped split, same features, same models, from Week 5

I already ran this comparison in Week 5, so this section re-runs and audits it rather than
re-inventing it — the "before" is the random train_test_split(..., stratify=y, random_state=42)
Week 3 also used, and the "after" is GroupShuffleSplit grouped by client_hash_id, so no client
appears in both train and test.

Results (F1 on the answered_away minority class, base rate 0.229):

| model | random split | grouped split | gap |
|---|---|---|---|
| Logistic Regression | 0.365 | 0.350 | 0.015 |
| Decision Tree | 0.057 | 0.014 | 0.043 |
| Random Forest | 0.136 | 0.110 | 0.026 |

Both splits clear Week 3's honest baseline F1 of 0.135 for Logistic Regression on both splits, and
Random Forest clears it narrowly on the random split (0.136) but falls just under it on the
grouped split (0.110). The random-to-grouped gap is small across all three models (0.015-0.043),
a different story than the framework video's own forest example (0.996 random down to 0.496
grouped) — that example had a much bigger drop, suggesting that model was leaning on something
client-specific. My gap is a weaker but real signal this feature set isn't leaning heavily on
memorized client identity, though 22 training clients and 8 test clients is still a small number
of groups to generalize from, so I'm stating this as observed on this split, not proven stable
across all possible client splits.

Note: re-running this cell produced slightly different random-split numbers than my first Week 5
run (0.365 vs 0.370 for Logistic Regression, 0.136 vs 0.149 for Random Forest), while the grouped
split numbers matched almost exactly. The likely cause is that DuckDB doesn't guarantee row order
without an explicit ORDER BY, so df can assemble in a different row order between sessions, which
shifts what train_test_split sees before its random_state=42 shuffle takes over. The grouped
split staying stable while the random split doesn't is itself informative — it's one more small
piece of evidence that the grouped result is the more trustworthy number to report, since it's
less sensitive to an incidental detail like row order.

Decision Tree collapsing to near-zero F1 on both splits is expected given a 22.9% minority class
and no class weighting on the tree — I'm not troubleshooting that here since Logistic Regression
and Random Forest are the models actually being compared to baseline.

In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

WEEK3_HONEST_F1 = 0.135
BASE_RATE = round(y.mean(), 3)

def run_and_score(model, X_tr, X_te, y_tr, y_te, name, split_name, scale=False):
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    f1 = f1_score(y_te, preds)
    return {'model': name, 'split': split_name, 'f1_answered_away': round(f1, 3)}

results = []

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
results.append(run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand,
                              'Logistic Regression', 'random', scale=True))
results.append(run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand, 'Decision Tree', 'random'))
results.append(run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand, 'Random Forest', 'random'))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
overlap = train_clients & test_clients
print(f"Client overlap in grouped split: {len(overlap)} (should be 0)")
assert len(overlap) == 0, "Grouped split leaked a client across train/test"

results.append(run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp,
                              'Logistic Regression', 'grouped', scale=True))
results.append(run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp, 'Decision Tree', 'grouped'))
results.append(run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp, 'Random Forest', 'grouped'))

results_df = pd.DataFrame(results)
print(f"\nBase rate (answered_away): {BASE_RATE}")
print(f"Week 3 honest F1 baseline (reference, random split): {WEEK3_HONEST_F1}")
print()
print(results_df.to_string(index=False))

pivot = results_df.pivot(index='model', columns='split', values='f1_answered_away')
pivot['gap'] = (pivot['random'] - pivot['grouped']).round(3)
print("\nRandom-to-grouped gap by model:")
print(pivot)

Client overlap in grouped split: 0 (should be 0)

Base rate (answered_away): 0.229
Week 3 honest F1 baseline (reference, random split): 0.135

              model   split  f1_answered_away
Logistic Regression  random             0.365
      Decision Tree  random             0.057
      Random Forest  random             0.136
Logistic Regression grouped             0.350
      Decision Tree grouped             0.014
      Random Forest grouped             0.110

Random-to-grouped gap by model:
split                grouped  random    gap
model                                      
Decision Tree          0.014   0.057  0.043
Logistic Regression    0.350   0.365  0.015
Random Forest          0.110   0.136  0.026


## 3. Leakage audit

Running the attack checklist from `hunting-leakage-and-validating` against my final Week 5
feature set: `gsc_impressions_prior30`, `gsc_clicks_prior30`, `gsc_avg_position_prior30`,
`word_count`, `gsc_ctr_prior30`, plus one-hot `content_type` and `main_intent`.

- **Timeline**: all five numeric features are prior-30-day window aggregates or static content
  properties, computed strictly before the last-30-day label window that defines
  `pattern_group`. None overlap the label window.
- **Label-derived / sibling columns**: `click_change_pct` (the actual leaky feature from Week 3)
  is confirmed absent from the final feature list. `gsc_ctr_prior30` is a ratio of two prior-
  window columns only (`gsc_clicks_prior30 / gsc_impressions_prior30`), so it doesn't touch
  `gsc_clicks_last30` or `gsc_impressions_last30` at all, which is what makes it pass where
  `click_change_pct` failed.
- **Product flags / existing-system scores**: none of the FlyRank Health Score, Optimization
  Flags, or Trend Direction columns are in the feature set. `pattern_group` itself is only ever
  used to build the label `y`, never as a feature — confirmed by checking `X.columns` doesn't
  contain it.
- **Grouped split**: done in Section 2, `client_hash_id` grouping, zero client overlap asserted.
- **Base rate printed next to every metric**: yes, 0.229 for answered_away, referenced in
  Section 2 next to every F1 score.
- **Top feature importance sanity check**: `gsc_avg_position_prior30` (0.234),
  `gsc_impressions_prior30` (0.229), and `gsc_ctr_prior30` (0.227) come out roughly tied as the
  top three in the Random Forest, with none towering over the others the way a leaked feature
  would (Week 3's `click_change_pct` before removal had that towering pattern — this doesn't).
  I'm treating this as a passed sanity check, not a celebration.
- **Note on `content_age_days`**: Section 1 of my Week 5 notebook describes adding
  `content_age_days` as a new feature, but the actual `feature_cols_numeric` list used in the
  model only contains the five features listed above — `content_age_days` isn't there. This is a
  documentation/code mismatch, not a leakage problem (the feature I described would have passed
  the decision-time test if it had been included), but I'm noting it here because catching gaps
  between what I said I did and what the code actually does is the exact muscle this assignment
  is asking me to build.

I also ran the deliberate-leak check the skill recommends: adding `click_change_pct` back into
the feature set and re-training, to confirm my test harness actually catches leakage rather than
just assuming it does.

In [5]:
# Deliberate-leak check: add back the known-leaky feature and confirm the score jumps.
# If it doesn't jump, the test harness itself can't be trusted.

df['click_change_pct'] = (
    (df['gsc_clicks_last30'] - df['gsc_clicks_prior30']) / df['gsc_clicks_prior30'] * 100
)

leaky_numeric = ['gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
                  'word_count', 'gsc_ctr_prior30', 'click_change_pct']
X_leaky = pd.get_dummies(df[leaky_numeric + feature_cols_categorical], dummy_na=True)
X_leaky[leaky_numeric] = X_leaky[leaky_numeric].fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

rf_leaky = RandomForestClassifier(n_estimators=200, random_state=42)
rf_leaky.fit(X_train_leak, y_train_leak)
preds_leaky = rf_leaky.predict(X_test_leak)
f1_leaky = f1_score(y_test_leak, preds_leaky)

# Compare against the honest random-forest random-split score from Section 2
honest_rf_random = results_df.query("model == 'Random Forest' and split == 'random'")['f1_answered_away'].iloc[0]

print(f"Honest Random Forest F1 (random split, no leak): {honest_rf_random}")
print(f"Random Forest F1 WITH click_change_pct added back in: {round(f1_leaky, 3)}")
print(f"Jump: {round(f1_leaky - honest_rf_random, 3)}")
print()
if f1_leaky > honest_rf_random + 0.1:
    print("Test harness confirmed working: adding a known-leaky feature produces a large jump,")
    print("same pattern as Week 3. Dropping click_change_pct again and keeping the honest features.")
else:
    print("No large jump — this would mean the test harness isn't sensitive to leakage and needs")
    print("investigation before trusting any other number in this notebook.")

# Feature importance check on the leaky version — confirms it's the leak driving the jump
importances_leaky = pd.Series(rf_leaky.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)
print("\nTop 5 feature importances with the leak present:")
print(importances_leaky.head(5))

Honest Random Forest F1 (random split, no leak): 0.136
Random Forest F1 WITH click_change_pct added back in: 0.722
Jump: 0.586

Test harness confirmed working: adding a known-leaky feature produces a large jump,
same pattern as Week 3. Dropping click_change_pct again and keeping the honest features.

Top 5 feature importances with the leak present:
click_change_pct            0.522479
gsc_avg_position_prior30    0.112578
gsc_impressions_prior30     0.109662
gsc_ctr_prior30             0.100671
word_count                  0.083500
dtype: float64


## 4. Claim rewrite

My boldest sentence, from Week 5's writeup:

Original: "Both [Logistic Regression and Random Forest] clear the honest F1 of 0.135 by a wide
margin on both splits — 0.370 random, 0.350 grouped."

Using the claim ladder from writing-honest-claims, this sentence sits between "a measured
comparison" and "a validated model that ranks/predicts out-of-sample" — I have a grouped,
out-of-sample split behind it, so it's not overclaiming into causal language, but "wide margin"
is doing some unearned drama work that the actual numbers don't need.

Rewrite: "On a client-grouped holdout, Logistic Regression scored F1=0.350 on the answered_away
class, against a base rate of 0.229 and Week 3's honest baseline of 0.135 — an observed
improvement over baseline on data the model's training clients never saw. Random Forest scored
F1=0.110 on the same grouped holdout, below Week 3's own Random Forest reference of 0.135, so the
improvement over baseline is specific to Logistic Regression in this comparison, not a property
of every model tried."

What changed: dropped "wide margin" (a drama word, not a measured one), named the actual split
type instead of leaving it implicit, put the base rate directly next to the F1 the way the
skill's checklist requires, and stopped implying all models improved when only two of three did
and one of those two only barely cleared the bar depending on which split you read. This is
decision-support language, not causal — it says the model looks worth using to help prioritize a
review queue, not that it will improve outcomes, since no A/B test sits behind any of this.

In [6]:
# No new computation needed — this section rewrites language, not numbers.
# Re-printing the exact figures the rewrite above depends on, so the claim and its
# evidence sit next to each other in the executed notebook.

print("Figures backing the Section 4 claim rewrite:")
print(f"Base rate (answered_away): {BASE_RATE}")
print(f"Week 3 honest F1 baseline: {WEEK3_HONEST_F1}")
print(results_df.query("model in ['Logistic Regression', 'Random Forest']").to_string(index=False))

Figures backing the Section 4 claim rewrite:
Base rate (answered_away): 0.229
Week 3 honest F1 baseline: 0.135
              model   split  f1_answered_away
Logistic Regression  random             0.365
      Random Forest  random             0.136
Logistic Regression grouped             0.350
      Random Forest grouped             0.110


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.